**IMPORTING LIBRARIES**

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

**LOADING AND INSPECTING DATASET**

In [2]:
def load_and_explore_dataset(file_path):
    print("=" * 60)
    print("LOAD AND EXPLORE DATE")
    print("=" * 60)

    df = pd.read_csv(file_path)
    print('shape of the dataset')
    print(df.shape)
    print('\ncheck missing')
    print(df.isnull().sum())
    print('\nFirst five row')
    print(df.head())
    print('\nDescriptive statistics')
    print(df.describe())
    print('\ndataset info')
    print(df.info())
    print('\nCrop Item distribution')
    print(df['item'].value_counts())
    print('\nDistribution of Years')
    print(df['year'].value_counts())

    return df

**ENCODING : PREPROCESSING**

In [18]:
## Preprocessing
def preprocessing_data(df):
    print('\n' + "=" * 60)
    print("PREPROCESSING DATA")
    print("=" * 60)


    df_preprocessed = df.copy()
    df_preprocessed = df_preprocessed.dropna()


    label_encoder = {}
    df_col = "area"
    le = LabelEncoder()
    df_preprocessed[df_col + "_encoded"] = le.fit_transform(df_preprocessed[df_col])
    label_encoder[df_col] = le
    print(f"\n{df_col} Encoded")
    for i, label in enumerate(le.classes_):
        print(f"{label}: {i}")

    ordinal_encoder = OrdinalEncoder(
        categories=[
            ["Potatoes", "Maize", "Wheat", "Rice, paddy", "Soybeans", "Sorghum", 
             "Sweet potatoes", "Cassava", "Yams", "Plantains and others"]
        ],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    df_preprocessed["item_encoded"] = (
        ordinal_encoder.fit_transform(df_preprocessed[["item"]])
    )

    print("\nItem Encoded")
    for i, label in enumerate(
        ordinal_encoder.categories_[0]
    ):
        print(f"{label}: {i}")
        
    print("Preprocessed data shape", df_preprocessed.shape)

    return df_preprocessed, label_encoder, ordinal_encoder

**CREATING DATA LABEL**

In [4]:
def featured_data(df_preprocessed):
    print('\n' + "=" * 60)
    print("FEATURED DATA")
    print("=" * 60)

    featured_columns = ["area_encoded", "year", "avg_rainfall_per_year", "pesticides_tonnes", "avg_temp", "item_encoded"]
    target_columns = ["crop_value"]

    X = df_preprocessed[featured_columns]
    y = df_preprocessed[target_columns]

    print("\nFeatures shape:", df_preprocessed[featured_columns].shape)
    print("\nTarget shape:", df_preprocessed[target_columns].shape)
    print("\nFeature columns:", featured_columns)

    return X, y, featured_columns

**SPLITING DATA: TRAINING AND TESTING**

In [5]:
def split_data(X, y, test_size=0.2, random_state=42):
    print('\n' + "=" * 60)
    print("SPLITING DATA")
    print("=" * 60)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    print("\nTraining set size:", X_train.shape[0])
    print("\nTest set size:", X_test.shape[0])

    print("\nTraining set crop value range {:.2f} - {:.2f}".format(
        float(y_train.values.min()), float(y_train.values.max())
    ))
    print("\nTest set crop value range {:.2f} - {:.2f}".format(
        float(y_test.values.min()), float(y_test.values.max())
    ))

    return X_train, X_test, y_train, y_test

**SCALING NUMERICAL DATA**

In [6]:
#making use of standard scaler
def scale_data(X_train, X_test):
    print('\n' + "=" * 60)
    print("SCALE DATA")
    print("=" * 60)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print("\nFeature scaled successfully")
    print("\nTraining set scaled shape:", X_train_scaled.shape)
    print("\nTesting set scaled shape:", X_test_scaled.shape)

    return X_train_scaled, X_test_scaled, scaler

**TRAINING MODEL**

In [7]:
def train_model(X_train_scale, y_train, featured_columns):
    print('\n' + "=" * 60)
    print("TRAIN MODEL")
    print("=" * 60)

    model = LinearRegression()
    model.fit(X_train_scale, y_train)

    print("\nModel Trained Successfully")
    print("\nModel Coefficient")

    for feature, coef in zip(featured_columns, model.coef_.ravel()):
        print(f" {feature}: {coef:.2f}")
    print(f"\nModel Intercept: {float(model.intercept_[0]):.2f}")

    return model

**RANDOM FOREST MODEL**

In [8]:
def train_random_forest_model(X_train_scale, y_train, featured_columns):
    print('\n' + "=" * 60)
    print("TRAIN RANDOM FOREST MODEL")
    print("=" * 60)

    rm_model = RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    )

    rm_model.fit(X_train_scale, y_train)
    print("\nRandom Forest Train Successfully!")
    print("Model estimator:", rm_model.estimator)
    print("Model max depth:", rm_model.max_depth)

    featured_importance = sorted(zip(featured_columns, rm_model.feature_importances_),
                                key=lambda x: x[1], reverse=True)

    for feature, importance in featured_importance:
        print(f"  {feature}: {float(importance):.4f}")

    return rm_model

**EVALUATING MODEL**

In [9]:
def evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test):
    print('\n' + "=" * 60)
    print("EVALUATE MODEL")
    print("=" * 60)

    # make prediction
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print('\n' + "=" * 60)
    print("MODEL PERFROMANCE")
    print("\nTraining Set")
    print(f" R2 score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")

    cv_score = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring="r2")
    print("\nCross Validation (5 Folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    metrics = {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score": cv_score
    }

    return metrics

**EVALUATING RANDOM FOREST MODEL**

In [10]:
def evaluate_rm_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test):
    print('\n' + "=" * 60)
    print("EVALUATE RM MODEL")
    print("=" * 60)

    # make prediction
    y_train_pred = rm_model.predict(X_train_scaled)
    y_test_pred = rm_model.predict(X_test_scaled)

    #calculate metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    print('\n' + "=" * 60)
    print("MODEL PERFROMANCE")
    print("\nTraining Set")
    print(f" R2 score: {train_r2:.4f}")
    print(f" RMSE: {train_rmse:.4f}")
    print(f" MAE: {train_mae:.4f}")

    print("\nTesting Set")
    print(f" R2 score: {test_r2:.4f}")
    print(f" RMSE: {test_rmse:.4f}")
    print(f" MAE: {test_mae:.4f}")

    cv_score = cross_val_score(rm_model, X_train_scaled, y_train, cv=5, scoring="r2")
    print("\nCross Validation (5 Folds)")
    print(f" R2 score: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    rm_metrics = {
        "train_r2": train_r2,
        "test_r2": test_r2,
        "train_rmse": train_rmse,
        "test_rmse": test_rmse,
        "train_mae": train_mae,
        "test_mae": test_mae,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "cv_score": cv_score
    }

    return rm_metrics

**SAVING MODEL ARTIFACT**

In [11]:
def save_model_artifact(rm_model, model, scaler, label_encoder, ordinal_encoder, featured_columns):
    print('\n' + "=" * 60)
    print("SAVING MODEL ARTIFACT")
    print("=" * 60)

    joblib.dump(rm_model, "rm_model_crop_yield_prediction.pkl")
    print("Crop Yield Pediction for rm model saved successfully.")

    joblib.dump(model, "crop_yield_prediction.pkl")
    print("Crop Yield Pediction model saved successfully.")

    joblib.dump(scaler, "scaler_features.pkl")
    print("Scaler features saved successfully.")

    joblib.dump(label_encoder, "label_encoder.pkl")
    print("Label encoder saved successfully.")

    joblib.dump(ordinal_encoder, "ordinal_encoder.pkl")
    print("Ordinal encoder saved successfully.")

    joblib.dump(featured_columns, "featured_columns.pkl")
    print("Featured columns saved successfully.")

**PREDICTION**

In [12]:
def predict_crop_value(area, year, rain_fall, pesticides, temp, item):
    model = joblib.load("crop_yield_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    ordinal_encoder = joblib.load("ordinal_encoder.pkl")
    featured_columns = joblib.load("featured_columns.pkl")

    try:
        area_encoded = label_encoder["area"].transform([area])[0]
        encoded_ordinal = ordinal_encoder.transform([[item]])
        item_encoded = encoded_ordinal[0, 0]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
        "area_encoded": area_encoded,
        "year": year,
        "rain_fall": rain_fall,
        "pesticides": pesticides,
        "temp": temp,
        "item_encoded": item_encoded,
    }

    features = np.array([[features_dict[col] for col in features_dict]])
    feature_scale = scaler.transform(features)

    predicted_price = float(model.predict(feature_scale).ravel()[0])
    
    return predicted_price

**RM PREDICTION**

In [13]:
def rm_model_predict_crop_value(area, year, rain_fall, pesticides, temp, item):
    rm_model = joblib.load("rm_model_crop_yield_prediction.pkl")
    scaler = joblib.load("scaler_features.pkl")
    label_encoder = joblib.load("label_encoder.pkl")
    ordinal_encoder = joblib.load("ordinal_encoder.pkl")
    featured_columns = joblib.load("featured_columns.pkl")

    try:
        area_encoded = label_encoder["area"].transform([area])[0]
        encoded_ordinal = ordinal_encoder.transform([[item]])
        item_encoded = encoded_ordinal[0, 0]
    except ValueError as e:
        return f"Unknown category - {e}"

    features_dict = {
        "area_encoded": area_encoded,
        "year": year,
        "rain_fall": rain_fall,
        "pesticides": pesticides,
        "temp": temp,
        "item_encoded": item_encoded,
    }

    features = np.array([[features_dict[col] for col in features_dict]])
    feature_scale = scaler.transform(features)

    predicted_price = float(rm_model.predict(feature_scale).ravel()[0])
    
    return predicted_price

**TESTING PREDICTION**

In [14]:
def test_prediction():
    print('\n' + "=" * 60)
    print("TESTING PREDICTION - LINEAR REGRESSION")
    print("=" * 60)

    #example 1 area, year, rain_fall, temp, item
    crop_value1 = predict_crop_value("Albania", 2015, 1000, 45.5, 20.22, "Maize")
    print("\n Maize in Albania 2015")
    print(f"Crop yield: {(crop_value1)}")
    
    # #example 2
    # crop_value1 = predict_crop_value("Albania", 2015, 1000, 45.5, 20.22, "Maize")
    # print("\n Maize in Albania 2015")
    # print(f"Crop yield: {(crop_value1)}")

    # #example 3
    crop_value3 = predict_crop_value("Angola", 1997, 1010.0, 40.0, 24.06, "Sweet Potatoes")
    print("\n Sweet Potatoes in Angola 1997")
    print(f"Crop yield: {(crop_value3)}")

**TESTING RM PREDICTION**

In [24]:
def test_rm_model_prediction():
    print('\n' + "=" * 60)
    print("TESTING PREDICTION - RANDOM FOREST REGRESSION")
    print("=" * 60)

    #example 1 area, year, rain_fall, temp, item
    crop_value1 = rm_model_predict_crop_value("Albania", 2015, 1000, 45.5, 20.22, "Maize")
    print("\n Maize in Albania 2015")
    print(f"Crop yield: {(crop_value1)}")
    
    # #example 2
    # crop_value1 = rm_model_predict_crop_value("Albania", 2015, 1000, 45.5, 20.22, "Maize")
    # print("\n Maize in Albania 2015")
    # print(f"Crop yield: {(crop_value1)}")

    # #example 3
    crop_value3 = rm_model_predict_crop_value("Angola", 1997, 1010.0, 40.0, 24.06, "Sweet Potatoes")
    print("\n Sweet Potatoes in Angola 1997")
    print(f"Crop yield: {(crop_value3)}")

**MAIN FUNCTION**

In [25]:
def main():
    file_path = 'cleaned_crop_yield_dataset.csv'

    ## load and explore datset
    df = load_and_explore_dataset(file_path)

    ##preprocessed
    df_preprocessed, label_encoder, ordinal_encoder = preprocessing_data(df)

    ##features data
    X, y, featured_columns = featured_data(df_preprocessed)

    ##split train and test data
    X_train, X_test, y_train, y_test = split_data(X, y)

    ##scaled 
    X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

    ##training model
    model = train_model(X_train_scaled, y_train, featured_columns)

    ##train random forest model 
    rm_model = train_random_forest_model(X_train_scaled, y_train, featured_columns)
    
    ##  evaluating model
    metrics = evaluate_model(model, X_train_scaled, X_test_scaled, y_train, y_test)

    ## evalute rm model
    rm_metrics = evaluate_rm_model(rm_model, X_train_scaled, X_test_scaled, y_train, y_test)

    ## saving model artifact
    save_model_artifact(rm_model, model, scaler, label_encoder, ordinal_encoder, featured_columns)

    ## test prediction
    test_prediction()

    ## rm model test prediction
    test_rm_model_prediction()

In [26]:
if __name__ == "__main__":
    main()

LOAD AND EXPLORE DATE
shape of the dataset
(25932, 7)

check missing
area                     0
year                     0
avg_rainfall_per_year    0
pesticides_tonnes        0
avg_temp                 0
item                     0
crop_value               0
dtype: int64

First five row
      area  year  avg_rainfall_per_year  pesticides_tonnes  avg_temp  \
0  Albania  1990                 1485.0              121.0     16.37   
1  Albania  1990                 1485.0              121.0     16.37   
2  Albania  1990                 1485.0              121.0     16.37   
3  Albania  1990                 1485.0              121.0     16.37   
4  Albania  1990                 1485.0              121.0     16.37   

          item  crop_value  
0        Maize       36613  
1     Potatoes       66667  
2  Rice, paddy       23333  
3      Sorghum       12500  
4     Soybeans        7000  

Descriptive statistics
               year  avg_rainfall_per_year  pesticides_tonnes      avg_temp  \
cou

C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)



Random Forest Train Successfully!
Model estimator: DecisionTreeRegressor()
Model max depth: 15
  item_encoded: 0.6203
  pesticides_tonnes: 0.1122
  avg_rainfall_per_year: 0.1006
  avg_temp: 0.0925
  area_encoded: 0.0470
  year: 0.0273

EVALUATE MODEL

MODEL PERFROMANCE

Training Set
 R2 score: 0.0282
 RMSE: 83036.2885
 MAE: 63503.7259

Testing Set
 R2 score: 0.0262
 RMSE: 85108.0470
 MAE: 65058.2648

Cross Validation (5 Folds)
 R2 score: [0.02849746 0.02648289 0.03127507 0.01788397 0.03161587]
 Cross validation mean: 0.0272
 Cross validation std: 0.0050

EVALUATE RM MODEL

MODEL PERFROMANCE

Training Set
 R2 score: 0.9930
 RMSE: 7070.9656
 MAE: 3270.6247

Testing Set
 R2 score: 0.9792
 RMSE: 12443.9132
 MAE: 5306.5635


C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\base.py:1403: Dat


Cross Validation (5 Folds)
 R2 score: [0.98073328 0.97789411 0.97752485 0.98076051 0.97666797]
 Cross validation mean: 0.9787
 Cross validation std: 0.0017

SAVING MODEL ARTIFACT
Crop Yield Pediction for rm model saved successfully.
Crop Yield Pediction model saved successfully.
Scaler features saved successfully.
Label encoder saved successfully.
Ordinal encoder saved successfully.
Featured columns saved successfully.

TESTING PREDICTION - LINEAR REGRESSION

 Maize in Albania 2015
Crop yield: 86612.07670111142

 Sweet Potatoes in Angola 1997
Crop yield: 67125.90139811137

TESTING PREDICTION - RANDOM FOREST REGRESSION


C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.


 Maize in Albania 2015
Crop yield: 15219.099955025296

 Sweet Potatoes in Angola 1997
Crop yield: 58392.5781904762


C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OrdinalEncoder was fitted with feature names
  warnings.warn(
C:\Users\MARYAM\Desktop\Crop-Yield-Prediction-Model\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
